# Séance 11 — Qualité, tests, production

## 🛟 Notebook « point de reprise »

**À quoi sert ce notebook ?** Si tu as manqué la séance précédente, si ton code
ne marche pas, ou si tu t'es perdu·e en route : **ouvre celui-ci**. Le code de
départ est déjà écrit et fonctionne. Tu n'as jamais besoin d'avoir réussi
l'exercice d'avant pour suivre celui d'aujourd'hui.

**Comment l'utiliser ?**
1. Exécute les cellules du haut sans les modifier (elles remettent tout en place).
2. Descends jusqu'aux cellules `# ✏️ À TOI DE JOUER`.
3. Écris ton code à la place des `...`.

**Raccourci** : `Maj + Entrée` exécute une cellule.

---


> ⚠️ Cette séance est une séance de **terminal**, pas de notebook.
> Ce fichier récapitule les commandes ; garde-le ouvert à côté.


## 1. uv — un seul outil à la place de cinq

```bash
uv init opportunitrack     # crée le squelette du projet
uv add fastapi uvicorn     # ajoute une dépendance ET l'installe
uv add --dev pytest ruff   # dépendances de développement
uv run pytest              # exécute sans activation manuelle
uv sync                    # reconstruit l'environnement à l'identique
```

`uv.lock` fige les versions exactes : « ça marche chez moi » devient
« ça marche partout », et c'est vérifiable.


## 2. Ruff — un seul outil à la place de quatre

```bash
ruff format .          # met en forme : le débat sur les espaces est clos
ruff check . --fix     # détecte et corrige ce qui peut l'être
```


## 3. Tester pour de vrai : fixtures et paramétrage

In [ ]:
# Contenu type de tests/test_modeles.py
from datetime import date, timedelta

import pytest


def est_urgente(jours: int) -> bool:
    return 0 <= jours < 7


@pytest.mark.parametrize("jours, attendu", [
    (-1, False),   # déjà passée
    (0, True),     # aujourd'hui
    (6, True),     # limite haute
    (7, False),    # juste au-delà du seuil
    (30, False),
])
def test_seuil_urgence(jours, attendu):
    """Un seul test, cinq cas. On teste les BORNES (6 et 7),
    jamais des valeurs confortables."""
    assert est_urgente(jours) is attendu


# Vérification rapide ici, dans le notebook :
for j, attendu in [(-1, False), (0, True), (6, True), (7, False)]:
    assert est_urgente(j) is attendu
print("✅ Les 4 cas passent")


## 4. Que tester en priorité ?

1. La **logique métier** (calculs, règles) — rentabilité maximale.
2. Les **cas limites** (liste vide, valeur nulle, date passée).
3. Les **bugs déjà rencontrés** — un bug corrigé sans test reviendra.
4. Les **routes d'API** en surface (statut + forme de la réponse).

**Ne pas tester** : les bibliothèques des autres, les getters triviaux,
la mise en forme.

⚠️ La couverture est un **indicateur**, pas un objectif. 60 % bien ciblés
valent mieux que 100 % d'assertions creuses.


## 5. Les secrets ne se versionnent jamais

Une clé d'API poussée sur un dépôt public est exploitée en quelques minutes.
Et **supprimer le fichier ne suffit pas** : l'historique Git conserve tout.

```python
from pydantic_settings import BaseSettings, SettingsConfigDict


class Config(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env")
    cle_api: str
    debug: bool = False
```

Avec `.env` dans `.gitignore` et un `.env.example` versionné **sans valeurs**.


---
# ✅ Checklist du livrable final

- [ ] Dépôt GitHub public avec README (installation, usage, captures)
- [ ] `pyproject.toml` + `uv.lock`
- [ ] `ruff check` et `ruff format --check` sans erreur
- [ ] Au moins 5 tests qui passent, dont un test d'API
- [ ] `.env.example` versionné, `.env` **jamais** versionné
- [ ] Un workflow CI avec badge vert
- [ ] Dockerfile fonctionnel
- [ ] API déployée et joignable (bonus)

---
# 🎤 Demo Day — 5 minutes

1. Ce que fait mon projet — 30 s, sans jargon.
2. Démonstration en direct — 2 min.
3. **La difficulté rencontrée et comment je l'ai résolue — 1 min 30.**
   *C'est le cœur de l'exercice : raconter un bug qui t'a pris trois heures
   est plus utile au groupe qu'une démonstration parfaite.*
4. Ce que j'ajouterais avec une semaine de plus — 1 min.
